# 03 — Self-Attention and Multi-Head Attention

    **Companion chapter:** `03-self-attention.md`

    ## Learning goals

    - Construct Q, K, and V projections.
- Implement scaled dot-product attention.
- Experimentally motivate score scaling.
- Separate matching from value retrieval.
- Implement multi-head attention and inspect tensor shapes.

    ## How to use this notebook

    Run the cells from top to bottom. Read the comments, change small values, and
    rerun the cell. Every notebook ends with practice prompts that can become
    GitHub issues, exercises, or discussion questions.

In [1]:
from __future__ import annotations

import math
import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## 1. Queries, keys, and values

We reproduce the three-token numerical example. Identity projection matrices
make `Q = K = V = X`, which lets us focus on the attention calculation.

In [2]:
tokens = ["Ali", "book", "read"]
X = np.array(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [1.0, 1.0],
    ]
)

W_Q = np.eye(2)
W_K = np.eye(2)
W_V = np.eye(2)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: (3, 2)
K shape: (3, 2)
V shape: (3, 2)


In [3]:
def stable_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / exp_x.sum(axis=axis, keepdims=True)


def scaled_dot_product_attention_np(
    q: np.ndarray,
    k: np.ndarray,
    v: np.ndarray,
    mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    d_k = q.shape[-1]
    scores = q @ k.T / np.sqrt(d_k)

    if mask is not None:
        scores = scores + mask

    weights = stable_softmax(scores, axis=-1)
    output = weights @ v
    return output, weights, scores


output, weights, scores = scaled_dot_product_attention_np(Q, K, V)

print("Scaled scores:\n", scores.round(3))
print("\nAttention weights:\n", weights.round(3))
print("\nContextualized output:\n", output.round(3))

assert np.allclose(weights.sum(axis=1), 1.0)

Scaled scores:
 [[0.707 0.    0.707]
 [0.    0.707 0.707]
 [0.707 0.707 1.414]]

Attention weights:
 [[0.401 0.198 0.401]
 [0.198 0.401 0.401]
 [0.248 0.248 0.503]]

Contextualized output:
 [[0.802 0.599]
 [0.599 0.802]
 [0.752 0.752]]


## 2. Read an attention row

Row `i` describes where token `i` retrieves information. The output is a weighted
mixture of value vectors, not merely an importance label.

In [4]:
query_index = 0

explanation = pd.DataFrame(
    {
        "key/value token": tokens,
        "attention weight": weights[query_index],
        "value dim 1": V[:, 0],
        "value dim 2": V[:, 1],
    }
)
explanation

,key/value token,attention weight,value dim 1,value dim 2
0,Ali,0.401112,1.0,0.0
1,book,0.197776,0.0,1.0
2,read,0.401112,1.0,1.0


## 3. Why divide by √dₖ?

With larger feature dimensions, unscaled dot products tend to become larger.
We compare the average maximum attention probability with and without scaling.

In [5]:
rows = []
for d_k in [4, 16, 64, 256, 1024]:
    maxima_unscaled = []
    maxima_scaled = []

    for _ in range(100):
        q = np.random.randn(1, d_k)
        k = np.random.randn(20, d_k)
        raw_scores = q @ k.T

        maxima_unscaled.append(stable_softmax(raw_scores, axis=-1).max())
        maxima_scaled.append(
            stable_softmax(raw_scores / np.sqrt(d_k), axis=-1).max()
        )

    rows.append(
        {
            "d_k": d_k,
            "mean max weight (unscaled)": np.mean(maxima_unscaled),
            "mean max weight (scaled)": np.mean(maxima_scaled),
        }
    )

scaling_results = pd.DataFrame(rows)
scaling_results

,d_k,mean max weight (unscaled),mean max weight (scaled)
0,4,0.392984,0.204713
1,16,0.666593,0.220779
2,64,0.792390,0.216081
3,256,0.931139,0.209869
4,1024,0.971514,0.228803


In [6]:
plt.plot(
    scaling_results["d_k"],
    scaling_results["mean max weight (unscaled)"],
    marker="o",
    label="Unscaled",
)
plt.plot(
    scaling_results["d_k"],
    scaling_results["mean max weight (scaled)"],
    marker="o",
    label="Scaled",
)
plt.xscale("log", base=2)
plt.xlabel("Key/query dimension d_k")
plt.ylabel("Mean maximum attention weight")
plt.title("Scaling prevents increasingly sharp softmax distributions")
plt.legend()
plt.show()

/tmp/ipykernel_679/3535992454.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Learned projections change matching and retrieval

`W_Q` and `W_K` affect which tokens match. `W_V` affects what information is
retrieved after the match.

In [7]:
W_Q = np.array([[1.0, 0.0], [0.5, 1.0]])
W_K = np.array([[0.5, 1.0], [1.0, 0.0]])
W_V = np.array([[1.0, 1.0], [0.0, 1.0]])

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

projected_output, projected_weights, _ = scaled_dot_product_attention_np(Q, K, V)

print("Projected weights:\n", projected_weights.round(3))
print("\nProjected output:\n", projected_output.round(3))

Projected weights:
 [[0.225 0.32  0.456]
 [0.332 0.195 0.473]
 [0.212 0.177 0.611]]

Projected output:
 [[0.68  1.456]
 [0.805 1.473]
 [0.823 1.611]]


## 5. Multi-head self-attention from scratch

Each head receives its own projected subspace. The head outputs are concatenated
and projected back to `d_model`.

In [8]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_projection = nn.Linear(d_model, d_model)
        self.k_projection = nn.Linear(d_model, d_model)
        self.v_projection = nn.Linear(d_model, d_model)
        self.output_projection = nn.Linear(d_model, d_model)

    def split_heads(self, tensor: torch.Tensor) -> torch.Tensor:
        batch, length, _ = tensor.shape
        tensor = tensor.view(batch, length, self.num_heads, self.head_dim)
        return tensor.transpose(1, 2)

    def forward(self, x: torch.Tensor):
        q = self.split_heads(self.q_projection(x))
        k = self.split_heads(self.k_projection(x))
        v = self.split_heads(self.v_projection(x))

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        weights = torch.softmax(scores, dim=-1)
        head_outputs = weights @ v

        batch, _, length, _ = head_outputs.shape
        concatenated = (
            head_outputs.transpose(1, 2)
            .contiguous()
            .view(batch, length, self.d_model)
        )
        output = self.output_projection(concatenated)
        return output, weights


x = torch.randn(2, 5, 8)
multihead = MultiHeadSelfAttention(d_model=8, num_heads=2)
output, weights = multihead(x)

print("Input:", tuple(x.shape))
print("Output:", tuple(output.shape))
print("Weights:", tuple(weights.shape), "(batch, heads, query, key)")
assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)))

Input: (2, 5, 8)
Output: (2, 5, 8)
Weights: (2, 2, 5, 5) (batch, heads, query, key)


In [9]:
head = 0
example_weights = weights[0, head].detach().numpy()

fig, ax = plt.subplots()
image = ax.imshow(example_weights)
ax.set_xlabel("Key positions")
ax.set_ylabel("Query positions")
ax.set_title(f"Randomly initialized attention head {head}")
fig.colorbar(image, ax=ax)
plt.show()

/tmp/ipykernel_679/167289546.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Practice

1. Set `W_V` to zero and explain the output.
2. Change one row of `X`; identify which score rows and columns change.
3. Add a fourth token and verify that the score matrix becomes `4 × 4`.
4. Change `num_heads` from 2 to 4 while keeping `d_model=8`.
5. Inspect whether two randomly initialized heads produce identical weights.